# ipynb/rgcnformer_cls_res.ipynb - RGCNFormer 分类结果 / RGCNFormer classification results

## 项目背景 / Background
RGCNFormer 分类结果
RGCNFormer classification results

## 功能模块 / Modules
- RGCNFormer 12 类的详细分类结果
- (详见各代码单元 / see code cells)

## 输入 / Inputs
- 上一阶段产物(.npy/.pt/.csv/.json)/ prior-stage outputs
- 内嵌常量与参数 / inline constants and params

## 输出 / Outputs
- 图表(内联显示) / figures (inline)
- 中间变量 / intermediate variables
- 导出文件(.png/.pdf/.csv) / exported files

## 数据流 / Data Flow
1. 加载数据 / Load data
2. 运行分析 / Run analysis
3. 渲染图表 / Render figures
4. 导出 / Export

## 相关文件 / Related Files
- 调用 / Calls: inference_modx_segmented.py、model/main_model.py
- 被调用 / Called by: 报告 / 论文 / report / paper

## 使用示例 / Usage Example
- 在 JupyterLab 中打开 / open in JupyterLab
- 逐单元运行 / run cells sequentially

## 作者 / Author
项目组 / Project Team

## 版本 / Version
1.0



In [ ]:
# # 安装ggplot2包（仅首次需要，安装过一次就不用再运行）
# install.packages("ggnewscale")

# # # # 加载ggplot2包（每次打开R脚本/会话都需要运行）
# # # library(ggplot2)
# # install.packages("dplyr")


In [ ]:
library(ggplot2)
library(dplyr)
library(tidyr)
library(patchwork)
library(ggnewscale)

# 1. 读取并准备数据
df <- read.csv("res.csv") 

# 模拟数据
# df <- data.frame(
#   Class = paste0("Class_", 1:10),
#   Acc = runif(10, 0.8, 0.95),
#   AUC = runif(10, 0.85, 0.98),
#   Precision = runif(10, 0.75, 0.9),
#   Recall = runif(10, 0.7, 0.9),
#   F1 = runif(10, 0.75, 0.92),
#   AUPRC = runif(10, 0.8, 0.96)
# )

metric_levels <- c("Acc","Precision","Recall", "AUC" ,  "F1","MCC", "AUPRC","Sn","Sp")
class_levels <- unique(df$Class)

plot_data <- df %>%
  select(Class, all_of(metric_levels)) %>%
  pivot_longer(cols = -Class, names_to = "Metric", values_to = "Value")

plot_data$Metric <- factor(plot_data$Metric, levels = metric_levels)
plot_data$Class <- factor(plot_data$Class, levels = rev(class_levels))

# 计算各 Metric 的平均值 (Total 栏)
metric_avg <- plot_data %>%
  group_by(Metric) %>%
  summarise(Avg_Value = mean(Value, na.rm = TRUE)) %>%
  mutate(Group = "Total")

# 2. 定义配色
bg_colors <- c("#F7F9F9", "#EAE7E1") 
bubble_palette <- c("#EAECEE", "#ABB2B9", "#85929E", "#5D6D7E", "#34495E")
total_palette <- c("#A9DFBF", "#7DCEA0")

# 3. 创建背景层数据
bg_data <- data.frame(
  Metric = factor(metric_levels, levels = metric_levels),
  xmin = seq_along(metric_levels) - 0.5,
  xmax = seq_along(metric_levels) + 0.5,
  bg_fill = rep(bg_colors, length.out = length(metric_levels))
)

# 4. 绘制左侧主图 (p1)
p1 <- ggplot() +
  geom_rect(data = bg_data, aes(xmin = xmin, xmax = xmax, ymin = -Inf, ymax = Inf, fill = bg_fill), 
            alpha = 1, show.legend = FALSE) +
  scale_fill_identity() + 
  
  geom_point(data = plot_data, aes(x = Metric, y = Class, size = Value, color = Value)) +
  
  # --- 修改点：label 乘以 100 并加上 % 符号 ---
  geom_text(data = plot_data, aes(x = Metric, y = Class, label = sprintf("%.2f", Value * 100)), 
            size = 2.8, color = "white", fontface = "bold") +
  
  scale_color_gradientn(colors = bubble_palette) +
  # 稍微调大 range 以适应更长的百分比文字
  scale_size_continuous(range = c(8, 20), limits = c(min(plot_data$Value)*0.8, 1)) +
  
  scale_x_discrete(position = "top", expand = expansion(mult = c(0, 0))) + 
  scale_y_discrete(expand = expansion(mult = c(0.05, 0.05))) +
  theme_minimal() +
  theme(
    axis.text.x = element_text(size = 12, face = "bold", color = "#4A4A4A"),
    axis.text.y = element_text(size = 12, color = "#4A4A4A"),
    axis.title = element_blank(),
    panel.grid = element_blank(),
    legend.position = "none",
    plot.margin = margin(t = 10, r = 0, b = 10, l = 10)
  )

# 5. 绘制底部 Total 栏 (p2)
p2 <- ggplot() +
  geom_rect(data = bg_data, aes(xmin = xmin, xmax = xmax, ymin = -Inf, ymax = Inf, fill = bg_fill), 
            alpha = 1, show.legend = FALSE) +
  scale_fill_identity() +
  new_scale_fill() +
  geom_tile(data = metric_avg, aes(x = Metric, y = Group, fill = Avg_Value), 
            width = 0.8, height = 0.6) +
  
  # --- 修改点：汇总栏同样改为百分比 ---
  geom_text(data = metric_avg, aes(x = Metric, y = Group, label = sprintf("%.2f%%", Avg_Value * 100)), 
            color = "white", fontface = "bold", size = 3.2) +
            
  scale_fill_gradientn(colors = total_palette) +
  scale_x_discrete(expand = expansion(mult = c(0, 0))) +
  theme_minimal() +
  theme(
    axis.text.x = element_blank(),
    axis.text.y = element_text(size = 12, face = "bold", color = "#4A4A4A"),
    axis.title = element_blank(),
    panel.grid = element_blank(),
    legend.position = "none",
    plot.margin = margin(t = 0, r = 0, b = 10, l = 10)
  )

# 6. 合并
final_plot <- p1 / p2 + plot_layout(heights = c(10, 1))



ggsave("final_percentage1.png", final_plot, width = 11, height = 12, dpi = 300)
print(final_plot)

In [ ]:
# library(ggplot2)
# library(dplyr)
# library(tidyr)
# library(patchwork)
# library(scales)

# # 1. 定义 9 种莫兰迪色相 (保持高级灰调)
# morandi_hues <- c(
#   "Acc" = "#768D99", "Precision" = "#8A9A8A", "Recall" = "#B28F8F",
#   "AUC" = "#B6A691", "F1" = "#7E8B8C", "MCC" = "#938BA1",
#   "AUPRC" = "#AD8B7D", "Sn" = "#88A096", "Sp" = "#A8A085"
# )

# # 2. 数据准备与颜色映射
# metric_levels <- c("Acc","Precision","Recall", "AUC" ,  "F1","MCC", "AUPRC","Sn","Sp")
# class_levels <- unique(df$Class)

# plot_data <- df %>%
#   select(Class, all_of(metric_levels)) %>%
#   pivot_longer(cols = -Class, names_to = "Metric", values_to = "Value") %>%
#   mutate(
#     Metric = factor(Metric, levels = metric_levels),
#     Class = factor(Class, levels = rev(class_levels))
#   ) %>%
#   group_by(Metric) %>%
#   mutate(
#     # 颜色映射：从浅灰到各指标的主色相
#     point_color = col_numeric(
#       palette = c("#EAECEE", morandi_hues[as.character(Metric[1])]), 
#       domain = c(0.3, 1) # 这里的 domain 建议稍微大于实际最小值
#     )(Value),
#     # 核心修改：如果数值小于 0.6，文字用深灰色，否则用白色
#     # text_color = ifelse(Value < 0.6, "#4A4A4A", "white")
#     text_color = "#4A4A4A"
#   ) %>%
#   ungroup()

# # 汇总栏颜色
# metric_avg <- plot_data %>%
#   group_by(Metric) %>%
#   summarise(Avg_Value = mean(Value, na.rm = TRUE)) %>%
#   mutate(avg_color = morandi_hues[as.character(Metric)], Group = "Total")

# # 3. 绘制主图 (p1)
# p1 <- ggplot() +
#   # 背景层保持不变
#   geom_rect(data = bg_data, aes(xmin = xmin, xmax = xmax, ymin = -Inf, ymax = Inf, fill = bg_fill), 
#             alpha = 1, show.legend = FALSE) +
#   scale_fill_identity() + 
  
#   # 绘制圆圈
#   geom_point(data = plot_data, aes(x = Metric, y = Class, size = Value, color = point_color)) +
  
#   # 绘制文字：使用 identity 映射动态颜色
#   geom_text(data = plot_data, aes(x = Metric, y = Class, 
#                                  label = sprintf("%.2f", Value * 100),
#                                  color = text_color), 
#             size = 2.8, fontface = "bold", show.legend = FALSE) +
  
#   scale_color_identity() + # 关键：让 point_color 和 text_color 生效
  
#   # 核心修改：调大 range 的起始值（12），保证最小圆圈也能包住文字
#   scale_size_continuous(range = c(8, 22), limits = c(0.35, 1)) +
  
#   scale_x_discrete(position = "top", expand = expansion(mult = c(0, 0))) + 
#   scale_y_discrete(expand = expansion(mult = c(0.05, 0.05))) +
#   theme_minimal() +
#   theme(
#     axis.text.x = element_text(size = 11, face = "bold", color = "#4A4A4A"),
#     axis.text.y = element_text(size = 11, color = "#4A4A4A"),
#     axis.title = element_blank(),
#     panel.grid = element_blank(),
#     legend.position = "none"
#   )

# # 4. 汇总栏 (p2)
# p2 <- ggplot() +
#   geom_rect(data = bg_data, aes(xmin = xmin, xmax = xmax, ymin = -Inf, ymax = Inf, fill = bg_fill), 
#             alpha = 1, show.legend = FALSE) +
#   scale_fill_identity() +
#   geom_tile(data = metric_avg, aes(x = Metric, y = Group, fill = avg_color), 
#             width = 0.8, height = 0.6) +
#   geom_text(data = metric_avg, aes(x = Metric, y = Group, label = sprintf("%.2f%%", Avg_Value * 100)), 
#             color = "white", fontface = "bold", size = 3.2) +
#   scale_fill_identity() +
#   scale_x_discrete(expand = expansion(mult = c(0, 0))) +
#   theme_minimal() +
#   theme(
#     axis.text.x = element_blank(),
#     axis.text.y = element_text(size = 11, face = "bold", color = "#4A4A4A"),
#     axis.title = element_blank(),
#     panel.grid = element_blank()
#   )

# # 5. 合并
# final_plot <- p1 / p2 + plot_layout(heights = c(10, 1))
# ggsave("final_percentage2.png", final_plot, width = 11, height = 12, dpi = 300)
# print(final_plot)

In [ ]:
# library(ggplot2)
# library(dplyr)
# library(tidyr)
# library(patchwork)
# library(scales)

# # 1. 定义 9 种莫兰迪色相
# morandi_hues <- c(
#   "Acc" = "#768D99", "Precision" = "#8A9A8A", "Recall" = "#B28F8F",
#   "AUC" = "#B6A691", "F1" = "#7E8B8C", "MCC" = "#938BA1",
#   "AUPRC" = "#AD8B7D", "Sn" = "#88A096", "Sp" = "#A8A085"
# )

# # 2. 数据准备
# metric_levels <- c("Acc","Precision","Recall", "AUC" ,  "F1","MCC", "AUPRC","Sn","Sp")
# class_levels <- unique(df$Class)

# plot_data <- df %>%
#   select(Class, all_of(metric_levels)) %>%
#   pivot_longer(cols = -Class, names_to = "Metric", values_to = "Value") %>%
#   mutate(
#     Metric = factor(Metric, levels = metric_levels),
#     Class = factor(Class, levels = rev(class_levels))
#   ) %>%
#   group_by(Metric) %>%
#   mutate(
#     point_color = col_numeric(
#       palette = c("#EAECEE", morandi_hues[as.character(Metric[1])]), 
#       domain = c(0.35, 1) 
#     )(Value),
#     text_color = "#4A4A4A" # 统一使用深灰字，也可根据上一版逻辑改回动态色
#   ) %>%
#   ungroup()

# metric_avg <- plot_data %>%
#   group_by(Metric) %>%
#   summarise(Avg_Value = mean(Value, na.rm = TRUE)) %>%
#   mutate(avg_color = morandi_hues[as.character(Metric)], Group = "Total")

# # 3. 绘制主图 (p1)
# p1 <- ggplot() +
#   geom_rect(data = bg_data, aes(xmin = xmin, xmax = xmax, ymin = -Inf, ymax = Inf, fill = bg_fill), 
#             alpha = 1, show.legend = FALSE) +
#   scale_fill_identity() + 
  
#   # 绘制圆圈
#   geom_point(data = plot_data, aes(x = Metric, y = Class, size = Value, color = point_color)) +
  
#   geom_text(data = plot_data, aes(x = Metric, y = Class, 
#                                  label = sprintf("%.2f", Value * 100),
#                                  color = text_color), 
#             size = 2.8, fontface = "bold", show.legend = FALSE) +
  
#   scale_color_identity() + 
  
#   # --- 核心修改：配置尺寸图例 ---
#   scale_size_continuous(
#     name = "Performance Value",
#     range = c(10, 22),       # 最小值设为10，确保38.80%也能包住文字
#     limits = c(0.35, 1), 
#     breaks = c(0.4, 0.6, 0.8, 1.0),
#     labels = c("40%", "60%", "80%", "100%")
#   ) +
  
#   # 强制给图例上色（使用莫兰迪中性色）
#   guides(
#     size = guide_legend(
#       override.aes = list(color = "#85929E"), # 使用中性莫兰迪灰
#       order = 1
#     )
#   ) +
  
#   scale_x_discrete(position = "top", expand = expansion(mult = c(0, 0))) + 
#   scale_y_discrete(expand = expansion(mult = c(0.05, 0.05))) +
#   theme_minimal() +
#   theme(
#     axis.text.x = element_text(size = 11, face = "bold", color = "#4A4A4A"),
#     axis.text.y = element_text(size = 11, color = "#4A4A4A"),
#     axis.title = element_blank(),
#     panel.grid = element_blank(),
#     # 图例样式微调
#     legend.position = "right",
#     legend.title = element_text(size = 10, face = "bold", color = "#4A4A4A"),
#     legend.text = element_text(size = 9, color = "#4A4A4A")
#   )

# # 4. 汇总栏 (p2)
# p2 <- ggplot() +
#   geom_rect(data = bg_data, aes(xmin = xmin, xmax = xmax, ymin = -Inf, ymax = Inf, fill = bg_fill), 
#             alpha = 1, show.legend = FALSE) +
#   scale_fill_identity() +
#   geom_tile(data = metric_avg, aes(x = Metric, y = Group, fill = avg_color), 
#             width = 0.8, height = 0.6) +
#   geom_text(data = metric_avg, aes(x = Metric, y = Group, label = sprintf("%.2f%%", Avg_Value * 100)), 
#             color = "white", fontface = "bold", size = 3.2) +
#   scale_fill_identity() +
#   scale_x_discrete(expand = expansion(mult = c(0, 0))) +
#   theme_minimal() +
#   theme(
#     axis.text.x = element_blank(),
#     axis.text.y = element_text(size = 11, face = "bold", color = "#4A4A4A"),
#     axis.title = element_blank(),
#     panel.grid = element_blank(),
#     legend.position = "none"
#   )

# # 5. 合并并收集图例
# final_plot <- p1 / p2 + 
#   plot_layout(heights = c(10, 1), guides = "collect") & 
#   theme(legend.justification = "bottom") # 让图例对齐上方

# # 保存
# ggsave("final_with_legend.png", final_plot, width = 12, height = 12, dpi = 300)
# print(final_plot)

In [ ]:
# install.packages("ragg")
# library(ragg)

In [ ]:
# 1. 安装并加载字体管理包
# if(!require(showtext)) install.packages("showtext")
library(showtext)

# --- 以下是你原本的代码，只需修改 theme 中的 family 参数 ---

library(ggplot2)
library(dplyr)
library(tidyr)
library(patchwork)
library(scales)
df <- read.csv("data/res.csv")

metric_levels <- c("Acc","Precision","Recall", "AUC" ,  "F1","MCC", "AUPRC","Sn","Sp")
class_levels <- unique(df$Class)

plot_data <- df %>%
  select(Class, all_of(metric_levels)) %>%
  pivot_longer(cols = -Class, names_to = "Metric", values_to = "Value")

plot_data$Metric <- factor(plot_data$Metric, levels = metric_levels)
plot_data$Class <- factor(plot_data$Class, levels = rev(class_levels))
# 计算各 Metric 的平均值 (Total 栏)
metric_avg <- plot_data %>%
  group_by(Metric) %>%
  summarise(Avg_Value = mean(Value, na.rm = TRUE)) %>%
  mutate(Group = "Total")

# 2. 定义配色
bg_colors <- c("#F7F9F9", "#EAE7E1") 
bubble_palette <- c("#EAECEE", "#ABB2B9", "#85929E", "#5D6D7E", "#34495E")
total_palette <- c("#A9DFBF", "#7DCEA0")
# 3. 创建背景层数据
bg_data <- data.frame(
  Metric = factor(metric_levels, levels = metric_levels),
  xmin = seq_along(metric_levels) - 0.5,
  xmax = seq_along(metric_levels) + 0.5,
  bg_fill = rep(bg_colors, length.out = length(metric_levels))
)


# 2. 注册微软雅黑 (确保名称与系统识别的一致)
# 如果你安装的是 .ttc，通常名字就是 "Microsoft YaHei"
font_add("YaHei", 
         regular = "/usr/share/fonts/truetype/myfonts/msyh.ttf", 
         bold = "/usr/share/fonts/truetype/myfonts/msyhbd.ttf")

# 2. 开启自动渲染
showtext_auto()

# 3. 这里的 dpi 建议与 ggsave 保持一致，防止预览和保存效果不一
showtext_opts(dpi = 300)



# 1. 定义 9 种莫兰迪色相
morandi_hues <- c(
  "Acc" = "#768D99", "Precision" = "#8A9A8A", "Recall" = "#B28F8F",
  "AUC" = "#B6A691", "F1" = "#7E8B8C", "MCC" = "#938BA1",
  "AUPRC" = "#AD8B7D", "Sn" = "#88A096", "Sp" = "#A8A085"
)

# 2. 数据准备
metric_levels <- c("Acc","Precision","Recall", "AUC" ,  "F1","MCC", "AUPRC","Sn","Sp")
class_levels <- unique(df$Class)

plot_data <- df %>%
  select(Class, all_of(metric_levels)) %>%
  pivot_longer(cols = -Class, names_to = "Metric", values_to = "Value") %>%
  mutate(
    Metric = factor(Metric, levels = metric_levels),
    Class = factor(Class, levels = rev(class_levels))
  ) %>%
  group_by(Metric) %>%
  mutate(
    point_color = col_numeric(
      palette = c("#EAECEE", morandi_hues[as.character(Metric[1])]), 
      domain = c(0.35, 1) 
    )(Value),
    text_color = "#4A4A4A" # 统一使用深灰字，也可根据上一版逻辑改回动态色
  ) %>%
  ungroup()

# 【删除了metric_avg汇总数据的计算，因为不需要Total栏了】

# 3. 绘制主图 (p1)
p1 <- ggplot() +
  geom_rect(data = bg_data, aes(xmin = xmin, xmax = xmax, ymin = -Inf, ymax = Inf, fill = bg_fill), 
            alpha = 1, show.legend = FALSE) +
  scale_fill_identity() + 
  
  # 绘制圆圈
  geom_point(data = plot_data, aes(x = Metric, y = Class, size = Value, color = point_color)) +
  
  geom_text(data = plot_data, aes(x = Metric, y = Class, 
                                 label = sprintf("%.2f", Value * 100),
                                 color = text_color), 
            size = 4, fontface = "bold", family = "YaHei", show.legend = FALSE) +
  
  scale_color_identity() + 
  
  # --- 核心修改：配置尺寸图例 ---
  scale_size_continuous(
    name = "Performance Value",
    range = c(10, 22),       # 最小值设为10，确保38.80%也能包住文字
    limits = c(0.35, 1), 
    breaks = c(0.4, 0.6, 0.8, 1.0),
    labels = c("40%", "60%", "80%", "100%")
  ) +
  
  # 强制给图例上色（使用莫兰迪中性色）
  guides(
    size = guide_legend(
      override.aes = list(color = "#85929E"), # 使用中性莫兰迪灰
      order = 1
    )
  ) +
  
  scale_x_discrete(position = "top", expand = expansion(mult = c(0, 0))) + 
  scale_y_discrete(expand = expansion(mult = c(0.05, 0.05))) +
  theme_minimal() +
  theme(
    # 修改：全局字体指定为 YaHei
    text = element_text(family = "YaHei"),
    axis.text.x = element_text(size = 12, face = "bold", color = "#4A4A4A"),
    axis.text.y = element_text(size = 12, color = "#4A4A4A"),
    axis.title = element_blank(),
    panel.grid = element_blank(),
    legend.position = "right",
    legend.title = element_text(size = 12, face = "bold", color = "#4A4A4A"),
    legend.text = element_text(size = 12, color = "#4A4A4A")
  )

# 【直接删除了p2汇总栏的全部绘制代码】

# 5. 最终图表（无需合并，直接使用p1，保留图例配置）
final_plot <- p1 & 
  theme(legend.justification = "bottom") # 让图例对齐上方

# 保存（可根据需要微调宽高，因为移除了汇总栏，高度可适当减小）
ggsave("png/final_without_total.pdf", final_plot, width = 12, height = 10, dpi = 300)
print(final_plot)

In [ ]:
library(showtext)
library(ggplot2)
library(dplyr)
library(tidyr)
library(scales)
library(ggforce)

# ==========================================
# 1. 基础配置
# ==========================================
font_add("YaHei", 
         regular = "/usr/share/fonts/truetype/myfonts/msyh.ttf", 
         bold = "/usr/share/fonts/truetype/myfonts/msyhbd.ttf")
showtext_auto()
showtext_opts(dpi = 300)

# ==========================================
# 2. 数据读取与清洗
# ==========================================

# A. 读取左侧数据 (res.csv)
df_cls_raw <- read.csv("data/res.csv", check.names = FALSE)
metric_levels_cls <- c("Acc","Precision","Recall", "AUC" ,  "F1","MCC", "AUPRC","Sn","Sp")

df_cls <- df_cls_raw %>%
  rename(Class_ID = `Mod Name`) %>%
  select(Class_ID, all_of(metric_levels_cls))

class_levels <- rev(unique(df_cls$Class_ID))
class_map <- setNames(1:length(class_levels), class_levels)

# B. 读取右侧数据 (rgcnformer_loc.csv)
df_loc_raw <- read.csv("data/rgcnformer_loc.csv", check.names = FALSE)
k_levels_raw <- c("Top-1", "Top-3", "Top-5", "Top-7", "Top-10", "Top-20", "Top-50")
k_levels_plot <- k_levels_raw 

df_loc <- df_loc_raw %>%
  rename(Class_ID = Name) %>%
  select(Class_ID, all_of(k_levels_raw))

# C. 读取统计背景数据 (statistic_loc.csv)
df_stat_raw <- read.csv("data/statistic_loc.csv", check.names = FALSE)
df_stat <- df_stat_raw %>%
  rename(Class_ID = Class) %>%
  select(Class_ID, Mean, Median, Mode)

# ==========================================
# 3. 数据转换
# ==========================================

# --- 3.1 左侧数据 (气泡) ---
# 【修改点】不再计算 point_color，而是准备 Metric 因子用于上色
d_left <- df_cls %>%
  pivot_longer(cols = -Class_ID, names_to = "Metric", values_to = "Value") %>%
  mutate(
    x_pos = as.numeric(factor(Metric, levels = metric_levels_cls)),
    y_pos = class_map[Class_ID],
    Metric = factor(Metric, levels = metric_levels_cls) # 确保因子顺序
  )

# --- 3.2 右侧背景 (统计条带) ---
offset_x <- length(metric_levels_cls)
k_values_map <- c(1, 3, 5, 7, 10, 20, 50)
get_k_index <- function(val) {
  idx <- which(k_values_map >= val)[1]
  if(is.na(idx)) idx <- length(k_values_map)
  return(idx)
}

colors_map <- c("Mean" = "#0077BB", "Median" = "#EE7733", "Mode" = "#33BBEE")
bg_rects <- list()

for(cid in class_levels) {
  stat_row <- df_stat %>% filter(Class_ID == cid)
  y_val <- class_map[cid]
  if(nrow(stat_row) > 0) {
    idx_mean <- get_k_index(stat_row$Mean)
    idx_median <- get_k_index(stat_row$Median)
    idx_mode <- get_k_index(stat_row$Mode)
    
    for(i in 1:length(k_levels_raw)) {
      x_center <- offset_x + i
      types_here <- c()
      if(i == idx_mean) types_here <- c(types_here, "Mean")
      if(i == idx_median) types_here <- c(types_here, "Median")
      if(i == idx_mode) types_here <- c(types_here, "Mode")
      
      if(length(types_here) == 0) next 
      
      n_seg <- 20
      cols <- if(length(unique(types_here)) == 1) {
        rep(colors_map[unique(types_here)], n_seg)
      } else {
        colorRampPalette(colors_map[unique(types_here)])(n_seg)
      }
      
      x_seq <- seq(x_center - 0.45, x_center + 0.45, length.out = n_seg + 1)
      for(k in 1:n_seg) {
        bg_rects[[length(bg_rects)+1]] <- data.frame(
          xmin = x_seq[k], xmax = x_seq[k+1],
          ymin = y_val - 0.45, ymax = y_val + 0.45,
          fill = cols[k],
          alpha = 0.5
        )
      }
    }
  }
}
d_bg_right <- if(length(bg_rects)>0) do.call(rbind, bg_rects) else data.frame()

# --- 构造隐形数据层 (用于图例) ---
dummy_legend_data <- data.frame(
  x = 1, y = 1, 
  fill_color = c("#0077BB", "#EE7733", "#33BBEE"),
  label = c("Mean", "Median", "Mode")
)

# --- 3.3 右侧数据 (圆环) ---
d_right <- df_loc %>%
  pivot_longer(cols = all_of(k_levels_raw), names_to = "Metric", values_to = "Value") %>%
  mutate(
    x_pos = offset_x + as.numeric(factor(Metric, levels = k_levels_raw)),
    y_pos = class_map[Class_ID],
    start_angle = 0,
    end_angle = Value * 2 * pi
  )

# --- 3.4 左侧背景 ---
d_bg_left <- data.frame(
  y = 1:length(class_levels),
  fill = ifelse((1:length(class_levels)) %% 2 == 0, "#F7F9F9", "#EAE7E1")
)

# ==========================================
# 4. 绘图 (合并输出)
# ==========================================

# 【修改点】定义莫兰迪色谱
morandi_hues <- c(
  "Acc" = "#768D99", "Precision" = "#8A9A8A", "Recall" = "#B28F8F",
  "AUC" = "#B6A691", "F1" = "#7E8B8C", "MCC" = "#938BA1",
  "AUPRC" = "#AD8B7D", "Sn" = "#88A096", "Sp" = "#A8A085"
)

all_labels <- c(metric_levels_cls, k_levels_plot)
all_breaks <- 1:length(all_labels)

p <- ggplot() +
  
  # 1. 隐形层 (Dummy Layer) - 专门用于生成统计分布图例
  geom_rect(data = dummy_legend_data, 
            aes(xmin=x, xmax=x, ymin=y, ymax=y, fill=fill_color), 
            alpha = 0, show.legend = TRUE) + 
  
  # 2. 左侧斑马纹
  geom_rect(data = d_bg_left, 
            aes(xmin = 0.5, xmax = length(metric_levels_cls) + 0.5, 
                ymin = y - 0.5, ymax = y + 0.5, fill = fill), 
            show.legend = FALSE) +
  
  # 3. 右侧统计背景 (渐变条)
  {if(nrow(d_bg_right) > 0) 
    geom_rect(data = d_bg_right, 
              aes(xmin=xmin, xmax=xmax, ymin=ymin, ymax=ymax, fill=fill, alpha=alpha), 
              show.legend = FALSE)} +
  
  # 4. 左侧气泡
  # 【修改点】将 color 映射到 Metric，不再是固定计算的 point_color
  geom_point(data = d_left, aes(x = x_pos, y = y_pos, size = Value, color = Metric)) +
  
  # 文字标签
  geom_text(data = d_left, aes(x = x_pos, y = y_pos, label = sprintf("%.1f", Value * 100)), 
            size = 3.0, fontface = "bold", family = "YaHei", color = "#4A4A4A") +
  
  # 5. 右侧圆环
  geom_arc(data = d_right, aes(x0 = x_pos, y0 = y_pos, r = 0.35, start = 0, end = 2*pi), 
           color = "#E8EAEB", linewidth = 2) +
  geom_arc(data = d_right, aes(x0 = x_pos, y0 = y_pos, r = 0.35, start = start_angle, end = end_angle), 
           color = "#4A4E69", linewidth = 1.2, lineend = "round") +
  geom_text(data = d_right, aes(x = x_pos, y = y_pos, label = sprintf("%.0f", Value * 100)), 
            size = 3.0, fontface = "bold", family = "YaHei", color = "#4A4E69") +
  
  # 6. 分割线
  geom_vline(xintercept = length(metric_levels_cls) + 0.5, color = "#5D6D7E", linetype = "dashed") +
  
  # --- 7. 比例尺与图例配置 (核心修改) ---
  
  # A. 统计分布图例 (Fill)
  scale_fill_identity(
    name = "Statistic Distribution",
    breaks = c("#0077BB", "#EE7733", "#33BBEE"),
    labels = c("Mean", "Median", "Mode"),
    guide = "legend"
  ) +
  
  # B. 性能指标颜色 (Color - Manual) -> 【修改点】应用莫兰迪色
  scale_color_manual(values = morandi_hues) +
  
  scale_alpha_identity() +
  
  # C. 气泡大小 (Size)
  scale_size_continuous(
    name = "Performance",
    range = c(6, 16),
    limits = c(0, 1.05),
    breaks = c(0.4, 0.6, 0.8, 1.0),
    labels = c("40%", "60%", "80%", "100%")
  ) +
  
  # D. Guides 控制
  guides(
    # 【修改点】取消 Metric 颜色的图例 (因为 X 轴已经有文字标签了，不需要额外图例)
    color = "none",
    
    # 【修改点】Performance (Size) 图例：去掉 gray override，改为深灰色，且 order=1
    size = guide_legend(override.aes = list(color = "#4A4A4A"), order = 1),
    
    # 统计分布图例：确保不透明
    fill = guide_legend(override.aes = list(alpha = 1), order = 2) 
  ) +
  
  scale_x_continuous(breaks = all_breaks, labels = all_labels, position = "top", expand = c(0, 0.5)) +
  scale_y_continuous(breaks = 1:length(class_levels), labels = class_levels, expand = c(0, 0.5)) +
  coord_fixed(ratio = 1, clip = "off") +
  
  theme_minimal() +
  theme(
    text = element_text(family = "YaHei"),
    panel.grid = element_blank(),
    axis.title = element_blank(),
    axis.text.x = element_text(size = 10, face = "bold", color = "#4A4A4A"),
    axis.text.y = element_text(size = 11, face = "bold", color = "#4A4A4A", hjust = 1),
    
    # 图例设置
    legend.position = "right",
    legend.title = element_text(face = "bold", size = 10),
    legend.text = element_text(size = 9),
    legend.margin = margin(l = 10),
    # 【修改点】去掉图例 Key 的背景方框
    legend.key = element_blank(),
    
    plot.margin = margin(20, 20, 20, 20)
  )

ggsave("png/merged_table_morandi_final.pdf", p, width = 18, height = 8, dpi = 300)